In [2]:
import pandas as pd
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("spam_SMS_dataset.csv")

In [4]:
df.info()
df.shape
df.isnull().sum()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   v1          5572 non-null   str    
 1   v2          5572 non-null   str    
 2   Unnamed: 2  0 non-null      float64
 3   Unnamed: 3  0 non-null      float64
 4   Unnamed: 4  0 non-null      float64
dtypes: float64(3), str(2)
memory usage: 674.1 KB


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
# df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)

In [6]:
df= df[["v1", "v2"]]

In [7]:
df["v1"] = df["v1"].map({"ham":0, "spam":1})

In [8]:
df.head()

,v1,v2
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
df.nunique()

v1       2
v2    5157
dtype: int64

In [10]:
df["text_words"] = df["v2"].str.split()
average_words = df["text_words"].str.len().mean()
print("Average words per row:", average_words)

Average words per row: 15.584170854271356


In [15]:
X = df["v2"]
y= df["v1"]

y[y == 0]
y[y == 1]

# Train Test Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =.2, random_state=42)

2       1
5       1
8       1
9       1
11      1
       ..
5537    1
5540    1
5547    1
5566    1
5567    1
Name: v1, Length: 747, dtype: int64

In [88]:
# Tokenize
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen = 50, padding = "post")
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=50, padding="post")

In [92]:
# Build model
model = Sequential([
    Embedding(input_dim = 5000, output_dim = 16, input_length=50 ),
    GlobalAveragePooling1D(),
    Dense(16, activation="relu"),
    Dense(1, activation = "sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [101]:
# Training the model
model.fit(X_train_seq, y_train, epochs=10, validation_data = (X_test_seq, y_test))

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9978 - loss: 0.0069 - val_accuracy: 0.9892 - val_loss: 0.0475
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9987 - loss: 0.0062 - val_accuracy: 0.9892 - val_loss: 0.0465
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9982 - loss: 0.0066 - val_accuracy: 0.9892 - val_loss: 0.0473
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9989 - loss: 0.0065 - val_accuracy: 0.9901 - val_loss: 0.0464
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9987 - loss: 0.0058 - val_accuracy: 0.9892 - val_loss: 0.0486
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9989 - loss: 0.0053 - val_accuracy: 0.9892 - val_loss: 0.0498
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9989 - loss: 0.0056 - val_accuracy: 0.9874 - val_loss: 0.0550
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9991 - loss: 0.0053 - val_accuracy: 0.

In [ ]:
# Evaluate the Model
loss, accuracy = model.evaluate(X_test_seq, y_test)
print(f" Test loss: {loss:.2%}")
print(f" Test accuracy: {accuracy:.2%}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9839 - loss: 0.0521 
 Test accuracy: 98.39%


In [97]:
# Save model
# model.save("model_spam_detector.h5")
model.save("model_spam_detector.keras")
with open("model_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)